### Homework 2 

In [1]:
import numpy as np
import pandas as pd

In [4]:
# ---------- 用網址載入 / Load by URL ----------
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)
cols = ['engine_displacement', 'horsepower', 'vehicle_weight',
        'model_year', 'fuel_efficiency_mpg']
df= df[cols]
df

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0
...,...,...,...,...,...
9995,2470,271.0,4690,2005,27.4
9996,2300,244.0,4370,1993,30.0
9997,2480,267.0,4590,2014,28.1
9998,2180,270.0,4450,2004,32.2


In [6]:
# ---------- Q1: 缺值欄位 / Column with NAs ----------
print(df.isnull().sum())

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [8]:
# ---------- Q2: horsepower 中位數 / Median ----------
print('horsepower median', df.horsepower.median())

horsepower median 254.0


Train Data Set

In [9]:
def split(seed):
    """60/20/20 切分 / split (same as lecture)"""
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    return (df.iloc[idx[:n_train]],
            df.iloc[idx[n_train:n_train + n_val]],
            df.iloc[idx[n_train + n_val:]])
 
def prep(d, fill):
    """補缺值並拆成 X, y / fill NAs, return X, y"""
    X = d.drop(columns='fuel_efficiency_mpg').fillna(fill).values
    y = d.fuel_efficiency_mpg.values
    return X, y
 
def train_linear(X, y, r=0.0):
    """Normal equation + 正規化 r / with regularization r"""
    X = np.c_[np.ones(len(X)), X]            # 加 bias 欄 / add bias column
    XTX = X.T @ X + r * np.eye(X.shape[1])   # r*I
    w = np.linalg.inv(XTX) @ X.T @ y
    return w[0], w[1:]
 
def predict(X, b, w):
    return b + X @ w
 
def rmse(y, p):
    return np.sqrt(((y - p) ** 2).mean())

In [10]:
# ---------- Q3: 0 vs mean ----------
df_train, df_val, df_test = split(42)
for name, fill in [('0', 0), ('mean', df_train.horsepower.mean())]:  # mean 只用 train / train only
    X, y = prep(df_train, fill)
    Xv, yv = prep(df_val, fill)
    b, w = train_linear(X, y)
    print('Q3', name, round(rmse(yv, predict(Xv, b, w)), 3))

Q3 0 2.205
Q3 mean 2.202


In [14]:
# ---------- Q3: 0 vs mean ----------
df_train, df_val, df_test = split(42)

results = {}  # 存兩種方法的 RMSE / store RMSE of each option

for name, fill in [('With 0', 0), ('With mean', df_train.horsepower.mean())]:
    # mean 只能用 train 算，避免偷看 val 資料 (data leakage)
    # mean must come from train only, to avoid leaking val data
    X, y = prep(df_train, fill)     # 訓練資料 / training data
    Xv, yv = prep(df_val, fill)     # 驗證資料 / validation data
    b, w = train_linear(X, y)       # 不加正規化 (r=0) / no regularization
    score = round(rmse(yv, predict(Xv, b, w)), 3)  # 四捨五入到 3 位 / round to 3 digits
    results[name] = score
    print('Q3', name, score)

# ---------- 判斷哪個比較好 / Decide which is better ----------
# RMSE = 預測誤差大小，越小代表預測越準
# RMSE = size of prediction error; smaller = more accurate
if results['With 0'] < results['With mean']:
    print('Answer: With 0')          # 補 0 的誤差較小 / filling 0 has lower error
elif results['With mean'] < results['With 0']:
    print('Answer: With mean')       # 補平均的誤差較小 / filling mean has lower error
else:
    print('Answer: Both are equally good')  # 四捨五入後一樣 / same after rounding

Q3 With 0 2.205
Q3 With mean 2.202
Answer: With mean


In [11]:
# ---------- Q4: 正規化 r / regularization ----------
X, y = prep(df_train, 0)
Xv, yv = prep(df_val, 0)
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    b, w = train_linear(X, y, r)
    print('Q4 r =', r, round(rmse(yv, predict(Xv, b, w)), 4))

Q4 r = 0 2.2053
Q4 r = 0.01 2.2058
Q4 r = 0.1 2.2241
Q4 r = 1 2.3492
Q4 r = 5 2.4094
Q4 r = 10 2.4195
Q4 r = 100 2.4292


In [12]:
# ---------- Q5: 不同 seed 的 std / std across seeds ----------
scores = []
for seed in range(10):
    tr, va, _ = split(seed)
    X, y = prep(tr, 0)
    Xv, yv = prep(va, 0)
    b, w = train_linear(X, y)
    scores.append(rmse(yv, predict(Xv, b, w)))
print('Q5 std:', round(np.std(scores), 3))

Q5 std: 0.029


In [13]:
# ---------- Q6: seed 9, train+val, r=0.001, test ----------
tr, va, te = split(9)
full = pd.concat([tr, va])
X, y = prep(full, 0)
Xt, yt = prep(te, 0)
b, w = train_linear(X, y, r=0.001)
print('Q6 test RMSE:', rmse(yt, predict(Xt, b, w)))

Q6 test RMSE: 2.235827747191731
